In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
# 下载训练集
train_dataset = datasets.MNIST(root='./',
                train=True,
                transform=transforms.ToTensor(),
                download=True)
# 下载测试集
test_dataset = datasets.MNIST(root='./',
               train=False,
               transform=transforms.ToTensor(),
               download=True)

In [3]:
# 批次大小
batch_size = 64

# 装载训练集
train_loader = DataLoader(dataset=train_dataset,
                      batch_size=batch_size,
                      shuffle=True)
# 装载测试集
test_loader = DataLoader(dataset=test_dataset,
                     batch_size=batch_size,
                     shuffle=True)

In [4]:
for i, data in enumerate(train_loader):
    # 获得数据和对应的标签
    inputs, labels = data
    print(inputs.shape)
    print(labels.shape)
    break

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [5]:
# 定义网络结构
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(1, 32, 5, 1, 2), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.conv2 = nn.Sequential(nn.Conv2d(32, 64, 5, 1, 2), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.fc1 = nn.Sequential(nn.Linear(64 * 7 * 7, 1000), nn.Dropout(p=0.4), nn.ReLU())
        self.fc2 = nn.Sequential(nn.Linear(1000, 10), nn.Softmax(dim=1))
        
    def forward(self, x):
        # ([64, 1, 28, 28])
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size()[0], -1)
        x = self.fc1(x)
        x = self.fc2(x)
        return x

In [6]:
LR = 0.0003
# 定义模型
model = Net()
# 定义代价函数
entropy_loss = nn.CrossEntropyLoss()
# 定义优化器
optimizer = optim.Adam(model.parameters(), LR)

In [7]:
def train():
    model.train()
    for i, data in enumerate(train_loader):
        # 获得数据和对应的标签
        inputs, labels = data
        # 获得模型预测结果，（64，10）
        out = model(inputs)
        # 交叉熵代价函数out(batch,C),labels(batch)
        loss = entropy_loss(out, labels)
        # 梯度清0
        optimizer.zero_grad()
        # 计算梯度
        loss.backward()
        # 修改权值
        optimizer.step()


def test():
    model.eval()
    correct = 0
    for i, data in enumerate(test_loader):
        # 获得数据和对应的标签
        inputs, labels = data
        # 获得模型预测结果
        out = model(inputs)
        # 获得最大值，以及最大值所在的位置
        _, predicted = torch.max(out, 1)
        # 预测正确的数量
        correct += (predicted == labels).sum()
    print("Test acc: {0}".format(correct.item() / len(test_dataset)))
    
    correct = 0
    for i, data in enumerate(train_loader):
        # 获得数据和对应的标签
        inputs, labels = data
        # 获得模型预测结果
        out = model(inputs)
        # 获得最大值，以及最大值所在的位置
        _, predicted = torch.max(out, 1)
        # 预测正确的数量
        correct += (predicted == labels).sum()
    print("Train acc: {0}".format(correct.item() / len(train_dataset)))

In [8]:
for epoch in range(0, 20):
    print('epoch:',epoch)
    train()
    test()

epoch: 0
Test acc: 0.9742
Train acc: 0.9729166666666667
epoch: 1
Test acc: 0.9813
Train acc: 0.9808333333333333
epoch: 2
Test acc: 0.9852
Train acc: 0.9841666666666666
epoch: 3
Test acc: 0.9877
Train acc: 0.9880833333333333
epoch: 4
Test acc: 0.9848
Train acc: 0.9867333333333334
epoch: 5
Test acc: 0.9897
Train acc: 0.9907333333333334
epoch: 6
Test acc: 0.9915
Train acc: 0.9922666666666666
epoch: 7
Test acc: 0.9898
Train acc: 0.993
epoch: 8
Test acc: 0.9923
Train acc: 0.9940166666666667
epoch: 9
Test acc: 0.9909
Train acc: 0.9939833333333333
epoch: 10
Test acc: 0.9913
Train acc: 0.9951333333333333
epoch: 11
Test acc: 0.991
Train acc: 0.99435
epoch: 12
Test acc: 0.9919
Train acc: 0.9961
epoch: 13
Test acc: 0.9927
Train acc: 0.9965666666666667
epoch: 14
Test acc: 0.9924
Train acc: 0.9961166666666667
epoch: 15
Test acc: 0.9924
Train acc: 0.9961666666666666
epoch: 16
Test acc: 0.9919
Train acc: 0.9967833333333334
epoch: 17
Test acc: 0.9917
Train acc: 0.9964333333333333
epoch: 18
Test acc: 0